# 🏖️ SyftBox — Data Scientist Tutorial

Welcome! This notebook walks you through using **SyftBox** as a **Data Scientist** — running on **Google Colab**.

## What is SyftBox?

SyftBox lets you run computations on a Data Owner's (DO) **private dataset** — **without ever seeing the raw data**. You write a Python script, submit it as a *job*, and the DO's machine runs it on the real data and sends back the results you wrote to an `outputs/` folder.

The cloud storage (Google Drive) you and the DO already use is the only infrastructure — there is no central server.

## How a job actually runs

This is the most important concept in the tutorial:

> Your job is a **standalone `.py` file** that runs in a fresh Python process on the Data Owner's machine. It shares **nothing** with your notebook — not your variables, not your imports, not your loaded data.

Two practical consequences:

1. **The script needs to be self-contained.** Whatever it uses, it has to import or define itself. (You'll see this in Step 9 — the `%%writefile` cell looks almost identical to the local-test cell, just slightly rearranged.)
2. **There's exactly one way to reach the data:** call `resolve_dataset_file_path("dataset_name")` from `syft_client`. The DO's runner makes this call return a path to the *real* private file. Anything you want back, you write into an `outputs/` folder.

The development loop is:

```
1. Test your analysis on the mock data (in this notebook)
2. %%writefile the same logic to a .py file
3. Submit the .py file as a job
4. Wait for the DO to approve and run it
5. Read the results
```

## What's in this tutorial

| Step | What you'll do |
|------|----------------|
| 1 | Install `syft-client` |
| 2 | Import the library |
| 3 | Configure your email and the DO's email |
| 4 | Log in (Google auth is automatic in Colab) |
| 5 | Add the DO as a peer (one-time setup) |
| 6 | Confirm the peer connection is active |
| 7 | Browse the DO's mock dataset |
| 8 | Write & test your analysis locally on mock data |
| 9 | Submit a simple job (stdlib only) |
| 10 | Submit a job with extra dependencies (pandas, scipy) |
| 11 | Submit an auto-approval job (`main.py` + `params.json`) |
| 12 | Track job status & retrieve results |
| 13 | Inspect failed jobs (stdout & stderr) |

## Before you start

You need:
1. **The Data Owner's email** — they'll have shared this with you.
2. **A Google account** — Colab handles authentication automatically.


---
## Step 1 — Install `syft-client`

Colab resets installed packages each time the runtime starts, so this needs to run every session.

We pin a specific version so this tutorial keeps working even as the library evolves.

In [ ]:
!uv pip install -q "syft-client==0.1.113"
print("✅ syft-client installed.")


---
## Step 2 — Import the library

We'll import `syft_client` as `sc` (matching the official docs convention).

In [ ]:
import syft_client as sc
print(f"✅ syft-client {sc.__version__} loaded.")

---
## Step 3 — ✏️ Enter your details

Set your own SyftBox email and the email of the Data Owner you want to work with.

| Variable | What to put here |
|----------|------------------|
| `MY_EMAIL` | Your Google account email — this becomes your SyftBox identity |
| `DO_EMAIL` | The Data Owner's email — used for peer connection, dataset access, and job submission |

In [ ]:
# ✏️ EDIT THESE TWO LINES
MY_EMAIL = "falcon.ronnie@gmail.com"      # ✏️ your SyftBox email
DO_EMAIL = "ronnie@openmined.org"    # ✏️ the Data Owner's email

# ── Validation ──────────────────────────────────────────────────────────
def _check_email(label, value, placeholder):
    if value == placeholder:
        return f"❌  {label} is still the placeholder — please edit it above."
    if "@" not in value or "." not in value.split("@")[-1]:
        return f"❌  {label} ({value!r}) doesn\'t look like an email address."
    return None

issues = [
    msg for msg in (
        _check_email("MY_EMAIL", MY_EMAIL, "your.email@gmail.com"),
        _check_email("DO_EMAIL", DO_EMAIL, "data.owner@example.com"),
    ) if msg
]

if issues:
    print("⚠️  Fix the following before continuing:\n")
    for msg in issues:
        print(" ", msg)
else:
    print("✅  Looks good!")
    print(f"    My email : {MY_EMAIL}")
    print(f"    DO email : {DO_EMAIL}")


---
## Step 4 — Log in to SyftBox

`sc.login_ds(...)` logs you in as a **Data Scientist** (`ds`). On Colab, your signed-in Google account is detected automatically — no token file or password needed.

> 🌐 The first time you run this you may see a Colab dialog asking permission to access your Drive. Click **Allow** — `syft-client` uses Drive as the transport channel for syncing files between you and the DO.

In [ ]:
client = sc.login_ds(email=MY_EMAIL)


---
## Step 5 — Add the Data Owner as a peer

A *peer* is someone you've established a two-way connection with. You need an active peer connection with the DO before you can submit jobs to them.

This step is a **one-time setup per Data Owner** — once accepted, the connection persists across notebook sessions, so you can skip Step 5 next time.

The DO will see your peer request and needs to approve it before the connection becomes active.

In [ ]:
client.add_peer(DO_EMAIL)


---
## Step 6 — Confirm the peer connection

Inspect `client.peers` to check the state of the connection.

| State | What it means |
|-------|---------------|
| ✅ active | Ready — you can submit jobs |
| ⏳ requested by me | The DO hasn't approved your request yet — ping them and re-run this cell later |
| 📩 requested by peer | They want to connect with you — call `client.approve_peer_request(DO_EMAIL)` |

In [ ]:
client.peers


---
## Step 7 — Browse the DO's mock dataset

The DO publishes each dataset in **two versions**:

- **Mock**: a synthetic look-alike with the same schema/columns as the real data. You can see this freely and use it to develop your code.
- **Private**: the real data. You never see it directly — it only exists on the DO's machine, and only the *results* of your approved jobs come back to you.

This step lists the datasets the DO has shared with you, then loads one for inspection.

In [ ]:
# List the datasets the DO has shared
datasets = client.datasets.get_all()

if not datasets:
    print("📭 No datasets available yet.")
    print("   Ask the DO whether they\'ve shared any datasets with your email.")
else:
    print(f"📦 {len(datasets)} dataset(s) available:\n")
    for i, ds in enumerate(datasets):
        print(f"  [{i}]  {ds.name}  (owner: {ds.owner})")


Pick one of the dataset names from the list above and assign it to `DATASET_NAME`.

In [ ]:
# ✏️ Set this to one of the dataset names printed above
DATASET_NAME = "beach_water_quality" #"your_dataset_name_here"   # ✏️

if not DATASET_NAME or DATASET_NAME == "your_dataset_name_here":
    print("❌  DATASET_NAME is still the placeholder — please edit it above.")
else:
    print(f"✅  Dataset selected: {DATASET_NAME!r}")


### Loading the mock data

`dataset.mock_files` returns a `list[Path]` pointing to all mock files for the dataset. Most datasets are a single CSV, in which case `mock_files[0]` is the file you want.

In [ ]:
import pandas as pd

dataset = client.datasets.get(DATASET_NAME, datasite=DO_EMAIL)
mock_files = dataset.mock_files

print(f"📁 {len(mock_files)} mock file(s) for {DATASET_NAME!r}:")
for f in mock_files:
    print(f"   • {f.name}  ({f.stat().st_size:,} bytes)")

# Load the first mock file
mock_df = pd.read_csv(mock_files[0])
print(f"\n📊 Shape: {len(mock_df):,} rows × {len(mock_df.columns)} columns")
mock_df.head()


### 🔗 Optional: handling URL columns (CSV pointing to remote resources)

---



*Skip this section if your dataset's CSV has no URL columns — jump to Step 8.*

A common dataset shape is a **CSV where one or more columns hold URLs** — links to images, PDFs, web pages, or other content that lives elsewhere. The CSV stays small and easy to share; the heavy content gets fetched on demand.

For example, an observations dataset might look like this:

| obs_id | animal | weight_kg | wiki_url | image_url |
|--------|--------|-----------|----------|-----------|
| MOCK001 | Cat | 1.16 | `https://example.com/wiki/Cat` | `https://example.com/images/cat.jpg` |
| MOCK002 | Dog | 38.87 | `https://example.com/wiki/Dog` | `https://example.com/images/dog.jpg` |
| MOCK003 | Mouse | 0.02 | `https://example.com/wiki/Mouse` | `https://example.com/images/mouse.jpg` |

Here, `dataset.mock_files` is just `[the_csv]` — a single file. The URLs are *data inside* the CSV; they're not separate files in the dataset.

The cell below defines two small helpers — `fetch_url_to_text` (for HTML/JSON/text) and `fetch_url_to_file` (for binary content like images and PDFs). Both handle network errors gracefully and return `None` on failure, so a single broken URL doesn't kill your whole analysis.

> 💡 The same helpers work in a job script — just **paste the function definitions into the job script** too, since scripts don't share state with the notebook. Note: the DO has to allow outbound network access for the URLs to actually fetch from inside a job.


In [ ]:
import requests
from pathlib import Path

def fetch_url_to_text(url: str, *, timeout: float = 10.0) -> str | None:
    """Fetch a URL as text. Returns None on failure (timeout, 4xx/5xx, DNS, ...)."""
    try:
        r = requests.get(
            url,
            timeout=timeout,
            headers={"User-Agent": "SyftBox-DS/1.0"},
        )
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"  ⚠️  {url}: {type(e).__name__}: {e}")
        return None


def fetch_url_to_file(url: str, out_path: Path, *, timeout: float = 30.0) -> Path | None:
    """Download a URL to a local file. Returns the path on success, None on failure.

    Useful for binary content (images, PDFs, audio) where you want to keep the
    bytes around for later analysis.
    """
    try:
        r = requests.get(
            url,
            timeout=timeout,
            headers={"User-Agent": "SyftBox-DS/1.0"},
            stream=True,
        )
        r.raise_for_status()
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return out_path
    except Exception as e:
        print(f"  ⚠️  {url}: {type(e).__name__}: {e}")
        return None


Run the helpers on your mock data. Set `URL_COLUMN` to the column in your CSV that holds the URLs you want to fetch.


In [ ]:
import pandas as pd

# ✏️ Adjust the column name to match your dataset
URL_COLUMN = "wiki_url"   # ✏️

# Re-fetch the dataset here so this section runs even if you skipped the
# "Loading the mock data" cell above (or restarted your kernel).
dataset = client.datasets.get(DATASET_NAME, datasite=DO_EMAIL)

# `dataset.mock_files[0]` is the single CSV — the URLs are columns inside it
df = pd.read_csv(dataset.mock_files[0])
print(f"📊 Loaded {len(df)} rows; first 3 URLs in {URL_COLUMN!r}:\n")
for url in df[URL_COLUMN].head(3):
    print(f"   • {url}")

print(f"\n🔗 Fetching the first URL as a sanity check...")
sample = df[URL_COLUMN].iloc[0]
content = fetch_url_to_text(sample)

if content is None:
    print(f"   (fetch failed — see warning above. Common causes: dummy URLs in mock data,")
    print(f"    network restricted in this environment, or the URL itself is dead.)")
else:
    print(f"   ✅ Got {len(content):,} chars from {sample}")
    print(f"   Preview: {content[:120]!r}")


---
## Step 8 — Test your analysis locally on mock data

**Always test on the mock data first**, before submitting a job. Mistakes are free here — once you submit, the DO has to review your script.

### The function you'll use everywhere: `resolve_dataset_file_path`

`syft_client.resolve_dataset_file_path("dataset_name")` is the single function that gets you to the data. It's clever about *which* data:

| Where it runs | What it returns |
|---------------|-----------------|
| In your notebook (with `client=client`) | Path to the **mock** file |
| Inside a submitted job script (no `client` arg) | Path to the **real private** file |

That's the whole trick: the same code works in both places. Develop against mock, submit, and the DO's runner swaps in the real data.

> 💡 If your dataset has multiple files, use `resolve_dataset_files_path` (plural) from `syft_client.utils` — it returns the full `list[Path]`. The singular version returns just the first file.

In [ ]:
import csv
import json
from syft_client import resolve_dataset_file_path

# In the notebook we pass client=client → returns the MOCK file path.
# In the job script (Step 9) we'll call this WITHOUT client= → returns the real file.
data_path = resolve_dataset_file_path(DATASET_NAME, client=client)

with open(data_path, newline="") as f:
    rows = list(csv.DictReader(f))

total = len(rows)
safe = sum(1 for r in rows if r["safe_to_swim"].strip().lower() == "true")
temps = [float(r["temperature_c"]) for r in rows]

result = {
    "total_readings":  total,
    "safe_days":       safe,
    "pct_safe":        round(safe / total * 100, 1),
    "avg_temperature": round(sum(temps) / len(temps), 2),
}

print("📊 Local test result (computed on MOCK data):")
print(json.dumps(result, indent=2))


---
## Step 9 — Submit a simple job

Now we'll save the same logic to a `.py` file and submit it. The DO's machine will re-run it, this time on the real private data.

### Two important differences vs. the local test

1. **No `client=` argument.** Inside a job script, `resolve_dataset_file_path("name")` runs on the DO's machine and returns the path to the *real* file. The DO's runner sets a `SYFTBOX_EMAIL` environment variable that the function uses to find the dataset.
2. **Write results to `outputs/`.** Anything you put in `./outputs/` gets shared back to you when the job finishes. Anything written elsewhere stays on the DO's machine.

### The `%%writefile` magic

`%%writefile filename.py` (must be the very first line of the cell) saves the cell's contents to a file in your Colab session. We use it to dump the analysis code to disk so we can submit it.

A "simple" job means it uses only the Python standard library — no extra `pip install` needed on the DO's side.

In [ ]:
%%writefile simple_analysis.py
"""Job script: runs on the DO\'s machine, on real private data."""
import csv
import json
import os
from syft_client import resolve_dataset_file_path

# No client= here → returns the path to the REAL data on the DO\'s machine.
data_path = resolve_dataset_file_path("beach_water_quality")   # ✏️

with open(data_path, newline="") as f:
    rows = list(csv.DictReader(f))

total = len(rows)
safe = sum(1 for r in rows if r["safe_to_swim"].strip().lower() == "true")
temps = [float(r["temperature_c"]) for r in rows]

result = {
    "total_readings":  total,
    "safe_days":       safe,
    "pct_safe":        round(safe / total * 100, 1),
    "avg_temperature": round(sum(temps) / len(temps), 2),
}

# Anything in outputs/ comes back to the Data Scientist
os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)


Now submit it.

Each submission gets a **timestamp suffix** appended to its name (e.g. `Beach Simple Analysis · 2026-05-04 18:42:13`). This keeps every run distinct so you can re-run this cell freely without colliding with previous submissions — and the timestamp makes it easy to spot which run is which when you list `client.jobs` in Step 12.

In [ ]:
from datetime import datetime

JOB_BASE_NAME = "Beach Simple Analysis"   # ✏️

# Append a timestamp so each submission has a unique, recognizable name.
# Format: "Beach Simple Analysis · 2026-05-04 18:42:13"
JOB_NAME = f"{JOB_BASE_NAME} · {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

client.submit_python_job(
    user=DO_EMAIL,
    code_path="simple_analysis.py",
    job_name=JOB_NAME,
)
print(f"⏳ Submitted as {JOB_NAME!r}. Run Step 12 to track its progress.")


---
## Step 10 — Submit a job with dependencies

Most real analyses need libraries beyond the standard library. Pass them as `dependencies=[...]` and the DO's runner will install them into a fresh, isolated virtual environment before your script runs.

> 💡 **Pin versions** for reproducibility (e.g. `"pandas==2.2.3"`). Jobs run on **Python 3.12**.

### 10a — Test it locally first

In [ ]:
import json
import pandas as pd
from scipy import stats
from syft_client import resolve_dataset_file_path

data_path = resolve_dataset_file_path(DATASET_NAME, client=client)
df = pd.read_csv(data_path)

safe   = df[df["safe_to_swim"] == True]
unsafe = df[df["safe_to_swim"] == False]

t_stat, p_value = stats.ttest_ind(safe["bacteria_cfu_ml"], unsafe["bacteria_cfu_ml"])
corr, _         = stats.pearsonr(df["ph"], df["bacteria_cfu_ml"])

result = {
    "total_readings":           len(df),
    "bacteria_ttest_statistic": round(float(t_stat), 4),
    "bacteria_ttest_pvalue":    round(float(p_value), 6),
    "ph_bacteria_correlation":  round(float(corr), 4),
}

print("📊 Local test result (on MOCK data):")
print(json.dumps(result, indent=2))


### 10b — Save it as a `.py` file

In [ ]:
%%writefile deps_analysis.py
"""Job script that uses pandas + scipy on the DO\'s real data."""
import json
import os
import pandas as pd
from scipy import stats
from syft_client import resolve_dataset_file_path

data_path = resolve_dataset_file_path("beach_water_quality")   # ✏️
df = pd.read_csv(data_path)

safe   = df[df["safe_to_swim"] == True]
unsafe = df[df["safe_to_swim"] == False]

t_stat, p_value = stats.ttest_ind(safe["bacteria_cfu_ml"], unsafe["bacteria_cfu_ml"])
corr, _         = stats.pearsonr(df["ph"], df["bacteria_cfu_ml"])

result = {
    "total_readings":           len(df),
    "bacteria_ttest_statistic": round(float(t_stat), 4),
    "bacteria_ttest_pvalue":    round(float(p_value), 6),
    "ph_bacteria_correlation":  round(float(corr), 4),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)


### 10c — Submit it with `dependencies=[...]`

In [ ]:
from datetime import datetime

JOB_BASE_NAME = "Beach Statistical Analysis"   # ✏️
JOB_NAME = f"{JOB_BASE_NAME} · {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

existing_names = {j.name for j in client.jobs}
i, candidate = 2, JOB_NAME
while candidate in existing_names:
    candidate = f"{JOB_NAME} ({i})"
    i += 1
JOB_NAME = candidate

client.submit_python_job(
    user=DO_EMAIL,
    code_path="deps_analysis.py",
    job_name=JOB_NAME,
    dependencies=["pandas==2.2.3", "scipy==1.14.1"],
)
print(f"⏳ Submitted as {JOB_NAME!r}. Run Step 12 to track its progress.")


---
## Step 11 — Submit a job for auto-approval

Manual review can be slow — every job needs the DO's eyes on it. To speed up repeat workflows, the DO can configure **`syft-bg`** (a background service) to **auto-approve** jobs whose `main.py` matches a pre-approved hash.

### The `main.py` + `params.json` convention

Auto-approval works best when the *code* doesn't change between runs but the *parameters* do. The convention is to split a job into two files:

| File | Role |
|------|------|
| `main.py` | The analysis. Reads `params.json` to know what to compute. **Must match the hash the DO whitelisted.** |
| `params.json` | The parameters for this run — filters, thresholds, column names, etc. The DO sees the params before each run. |

You don't have to do anything special on the DS side — just submit `main.py` and `params.json` together. If the DO has set up auto-approval for that specific `main.py`, the job runs without manual review.

### Submitting multiple files: pass a folder

`client.submit_python_job(code_path=...)` takes either a single `.py` file path **or a folder path**. To submit `main.py` + `params.json` together, put them in a folder and pass the folder.

### 11a — Set up the job folder

In [ ]:
import os
import json

JOB_DIR = "auto_job"
os.makedirs(JOB_DIR, exist_ok=True)

# ✏️ EDIT THESE PARAMETERS for each run — main.py reads them from params.json
params = {
    "dataset":     DATASET_NAME,                              # ✏️
    "metric":      "pct_safe",                                # ✏️
    "description": "Percentage of days safe to swim",         # ✏️
}

with open(f"{JOB_DIR}/params.json", "w") as f:
    json.dump(params, f, indent=2)

print(f"✅ Wrote {JOB_DIR}/params.json")
print(json.dumps(params, indent=2))


### 11b — Write `main.py`

This is the file the DO whitelists. Once approved, you can re-run the same `main.py` with different `params.json` and it'll auto-run.

In [ ]:
%%writefile auto_job/main.py
"""Auto-approval job: reads params.json, computes the requested metric."""
import csv
import json
import os
from syft_client import resolve_dataset_file_path

with open("params.json") as f:
    params = json.load(f)

data_path = resolve_dataset_file_path(params["dataset"])

with open(data_path, newline="") as f:
    rows = list(csv.DictReader(f))

total = len(rows)
safe  = sum(1 for r in rows if r["safe_to_swim"].strip().lower() == "true")

result = {
    "dataset":        params["dataset"],
    "metric":         params["metric"],
    "description":    params["description"],
    "total_readings": total,
    "pct_safe":       round(safe / total * 100, 1),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)


### 11c — Submit the folder

When `code_path` is a folder, `submit_python_job` auto-detects `main.py` as the entrypoint.

In [ ]:
from datetime import datetime

JOB_BASE_NAME = "Beach Quick Stats"   # ✏️
JOB_NAME = f"{JOB_BASE_NAME} · {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

existing_names = {j.name for j in client.jobs}
i, candidate = 2, JOB_NAME
while candidate in existing_names:
    candidate = f"{JOB_NAME} ({i})"
    i += 1
JOB_NAME = candidate

client.submit_python_job(
    user=DO_EMAIL,
    code_path="auto_job",          # folder containing main.py + params.json
    job_name=JOB_NAME,
)
print(f"⏳ Submitted as {JOB_NAME!r}. If the DO has auto-approval enabled for this main.py, it will run without waiting for manual review.")


---
## Step 12 — Track jobs & retrieve results

Re-run this section as often as needed — `client.jobs` auto-syncs with the DO's machine each time you access it.

### Job status reference

Jobs progress through these states in order. Re-run the cell below to refresh.

| Status | Meaning |
|--------|---------|
| 📥 `received` | The DO's machine has received your files but hasn't validated them yet. Wait a few seconds and re-run. |
| ⏳ `pending` | Validated and waiting for the DO to review. May take a while — the DO has to look at your script. |
| 🔄 `approved` | DO approved it. Queued to run. |
| ⚙️ `running` | Currently executing on the DO's machine. |
| ✅ `done` | Finished. **Jump to the next cell to read the results.** |
| 🚫 `rejected` | The DO declined the job. They may have replied with a reason — ask them. |
| ❌ `failed` | Your script crashed. See Step 13 for stdout/stderr. |

In [ ]:
if not client.jobs:
    print("📭 No jobs found yet.")
else:
    print(f"📋 Your jobs ({len(client.jobs)} total):\n")
    for i, job in enumerate(client.jobs):
        print(f"  [{i}]  {job.status:9s}  {job.name}")


### Read the results from a completed job

Pick a job by its index from the list printed in the previous cell, then run this cell to view its results.

In [ ]:
import json
import os
from IPython.display import display, Markdown

JOB_INDEX = 5 #0   # ✏️ index from the list above

if JOB_INDEX < 0 or JOB_INDEX >= len(client.jobs):
    print(f"❌ JOB_INDEX={JOB_INDEX} is out of range. Pick a number between 0 and {len(client.jobs) - 1}.")
else:
    job = client.jobs[JOB_INDEX]
    display(Markdown(f"### 📊 `[{JOB_INDEX}]` {job.name}  —  *{job.status}*"))

    if job.status != "done":
        print(f"⏳ This job isn't finished yet (status: {job.status!r}).")
        print("   Re-run Step 12 to refresh, or jump to Step 13 if it failed.")
    elif not job.output_paths:
        print("   (no output files)")
    else:
        for path in job.output_paths:
            display(Markdown(f"📁 `{path.name}`"))
            ext = os.path.splitext(path)[-1].lower()
            try:
                if ext == ".json":
                    with open(path) as f:
                        print(json.dumps(json.load(f), indent=2))
                elif ext == ".csv":
                    import pandas as pd
                    display(pd.read_csv(path).head(10))
                else:
                    with open(path) as f:
                        print(f.read())
            except Exception as e:
                print(f"   ⚠️  Could not read: {e}")


---
## Step 13 — Inspect a failed job

When a job has status `failed`, the script raised an exception on the DO's machine. The DO's runner captures stdout and stderr from your script and exposes them as `job.stdout` and `job.stderr`.

Pick the failed job by its index from the list in Step 12. The traceback is almost always in **`stderr`**.

In [ ]:
JOB_INDEX = 0   # ✏️ index from the list in Step 12

if JOB_INDEX < 0 or JOB_INDEX >= len(client.jobs):
    print(f"❌ JOB_INDEX={JOB_INDEX} is out of range. Pick a number between 0 and {len(client.jobs) - 1}.")
else:
    job = client.jobs[JOB_INDEX]
    print(f"Job       : [{JOB_INDEX}] {job.name}")
    print(f"Status    : {job.status}")
    print(f"Submitted : {job.submitted_at}")

    if job.status != "failed":
        print(f"\nℹ️  This job didn't fail (status: {job.status!r}).")
        print("   stdout/stderr are still shown below if available.")

    stderr = str(job.stderr)
    stdout = str(job.stdout)

    print("\n--- stderr (traceback usually here) ---")
    print(stderr if stderr.strip() else "(empty)")

    print("\n--- stdout ---")
    print(stdout if stdout.strip() else "(empty)")


### Common reasons jobs fail

- **Referenced a notebook variable** (`client`, `mock_df`, `dataset`, etc.) — the script is a fresh Python process and only sees what's in the file itself.
- **Passed `client=client` to `resolve_dataset_file_path` inside the job script** — that argument is for local testing in the notebook only. In the job script, call it with just the dataset name.
- **Forgot a dependency** — every non-stdlib import needs to be listed in `dependencies=[...]`.
- **Wrote results outside `outputs/`** — only files written to `./outputs/` come back to you.
- **Syntax error or undefined name** — testing locally first (Step 8 / 10a) catches almost all of these before submission.

---
## 🎉 You're done!

You've successfully:
- ✅ Connected to SyftBox as a Data Scientist from Colab
- ✅ Browsed the DO's mock dataset
- ✅ Tested an analysis locally on mock data
- ✅ Submitted a stdlib-only job
- ✅ Submitted a job with extra dependencies
- ✅ Submitted an auto-approval job (`main.py` + `params.json`)
- ✅ Retrieved results and inspected failed-job logs

## On future Colab sessions

You'll need to re-run **Step 1** (install) and **Step 4** (login) every session — Colab resets installed packages and processes. Steps 5 (add peer) is one-time per Data Owner.

A typical follow-up session looks like:
```
Step 1 → Step 2 → Step 3 → Step 4 → Step 8 → Step 9/10/11 → Step 12
```

## Resources

- [Data Scientist guide](https://www.mintlify.com/OpenMined/syft-client/guides/data-scientist)
- [`syft-job` package docs](https://www.mintlify.com/OpenMined/syft-client/packages/syft-job)
- [`syft-bg` (auto-approval) docs](https://www.mintlify.com/OpenMined/syft-client/packages/syft-bg)
- [Full API reference](https://www.mintlify.com/OpenMined/syft-client/api/client/login)
- [OpenMined Community Slack](https://slack.openmined.org)
